## Note on data inputs

This notebook documents the **analysis method**. The raw licensed-facility inputs it reads (California Community Care Licensing / CDPH files containing facility administrator names, telephone numbers, and complaint histories) are **not distributed** with this repository, to protect vulnerable populations. De-identified and aggregated outputs are included; see `docs/DATA_AVAILABILITY.md`.

_As a result, this notebook is provided for transparency of method and will not run end-to-end against this public repository alone._

In [ ]:
from pathlib import Path

import pandas as pd
import geopandas as gpd

ROOT = Path("..")

DATA = ROOT / "data"

RAW = DATA / "raw"
PROCESSED = DATA / "processed"
OUTPUTS = ROOT / "outputs"

In [ ]:
eaton = pd.read_csv(
    PROCESSED / "calfire" / "eaton_fire_only.csv"
)

In [ ]:
import os

os.listdir("../data/processed/calfire")

In [ ]:
from pathlib import Path

import pandas as pd
import geopandas as gpd
import os

ROOT = Path("..")

DATA = ROOT / "data"

RAW = DATA / "raw"
PROCESSED = DATA / "processed"
OUTPUTS = ROOT / "outputs"

In [ ]:
os.listdir(PROCESSED)

In [ ]:
os.listdir(PROCESSED)

In [ ]:
from pathlib import Path

import pandas as pd
import geopandas as gpd
import os

ROOT = Path("..")
DATA = ROOT / "data"
RAW = DATA / "raw"
PROCESSED = DATA / "processed"

In [ ]:
os.listdir(PROCESSED)

In [ ]:
os.listdir(PROCESSED / "calfire")

In [ ]:
eaton = pd.read_csv(
    PROCESSED / "calfire" / "eaton_fire_only.csv"
)

eaton.head()

In [ ]:
calfire_school_assessed = eaton[
    eaton["* Structure Type"] == "School"
].copy()

len(calfire_school_assessed)

In [ ]:
calfire_school_assessed_export = (
    calfire_school_assessed[
        [
            "APN (parcel)",
            "Site Address (parcel)",
            "Latitude",
            "Longitude",
            "* Damage"
        ]
    ]
    .drop_duplicates()
)

calfire_school_assessed_export.to_csv(
    PROCESSED / "schools" / "calfire_assessed_school_structures.csv",
    index=False
)

In [ ]:
calfire_school_major = calfire_school_assessed[
    calfire_school_assessed["* Damage"].isin([
        "Major (25-50%)",
        "Destroyed (>50%)"
    ])
].copy()

len(calfire_school_major)

In [ ]:
calfire_school_major_export = (
    calfire_school_major[
        [
            "APN (parcel)",
            "Site Address (parcel)",
            "Latitude",
            "Longitude",
            "* Damage"
        ]
    ]
    .drop_duplicates()
)

calfire_school_major_export.to_csv(
    PROCESSED / "schools" / "calfire_school_major_destroyed.csv",
    index=False
)

In [ ]:
school_major_percent = (
    len(calfire_school_major)
    / len(calfire_school_assessed)
) * 100

round(school_major_percent, 1)

In [ ]:
school_summary = pd.DataFrame({
    "category": ["School"],
    "total_assessed": [len(calfire_school_assessed)],
    "major_destroyed": [len(calfire_school_major)],
    "percent_major_destroyed": [round(school_major_percent, 1)]
})

school_summary.to_csv(
    PROCESSED / "schools" / "school_impact_summary.csv",
    index=False
)

school_summary

In [ ]:
schools_external = pd.read_csv(
    RAW / "schools" / "Schools_Colleges_and_Universities_-423728542109668590.csv"
)

schools_external.head()

In [ ]:
schools_external["COLUMN_NAME"].value_counts()

In [ ]:
schools_external["Category2"].value_counts()

In [ ]:
schools_external["Category3"].value_counts()

In [ ]:
schools_filtered = schools_external[
    (
        schools_external["Category2"].isin([
            "Elementary Schools",
            "High Schools",
            "Intermediate/Middle/Junior High Schools",
            "Elementary-High Combination Schools"
        ])
    )
    &
    (
        schools_external["Category3"].isin([
            "Public Schools",
            "Private Schools",
            "Charter Schools"
        ])
    )
].copy()

In [ ]:
len(schools_filtered)

In [ ]:
schools_filtered["Category2"].value_counts()

In [ ]:
schools_filtered.columns.tolist()

In [ ]:
schools_gdf = gpd.GeoDataFrame(
    schools_filtered,
    geometry=gpd.points_from_xy(
        schools_filtered["Longitude"],
        schools_filtered["Latitude"]
    ),
    crs="EPSG:4326"
)

In [ ]:
schools_gdf = schools_gdf.to_crs(
    perimeter.crs
)

In [ ]:
perimeter = gpd.read_file(
    RAW / "perimeters" / "eaton_perimeter.geojson"
)

In [ ]:
perimeter = gpd.read_file(
    RAW / "perimeters" / "eaton_perimeter.geojson"
)

In [ ]:
perimeter.head()

In [ ]:
perimeter.crs

In [ ]:
schools_gdf = schools_gdf.to_crs(
    perimeter.crs
)

In [ ]:
schools_in_fire = gpd.sjoin(
    schools_gdf,
    perimeter,
    predicate="intersects"
)

In [ ]:
len(schools_in_fire)

In [ ]:
schools_in_fire[
    [
        "Name",
        "Category2",
        "Category3",
        "Address Line 1",
        "City"
    ]
]

In [ ]:
schools_in_fire.to_csv(
    PROCESSED / "schools" / "schools_inside_eaton_perimeter.csv",
    index=False
)

In [ ]:
len(calfire_school_assessed)

In [ ]:
len(schools_in_fire)

In [ ]:
len(calfire_school_assessed)

In [ ]:
schools_in_fire[
    [
        "Name",
        "Category2",
        "Category3",
        "Address Line 1",
        "City"
    ]
]

In [ ]:
calfire_school_parcels = (
    calfire_school_assessed[
        [
            "APN (parcel)",
            "Site Address (parcel)",
            "* Damage"
        ]
    ]
    .drop_duplicates(subset=["APN (parcel)"])
)

len(calfire_school_parcels)

In [ ]:
calfire_school_major_parcels = (
    calfire_school_major[
        [
            "APN (parcel)",
            "Site Address (parcel)",
            "* Damage"
        ]
    ]
    .drop_duplicates(subset=["APN (parcel)"])
)

len(calfire_school_major_parcels)

In [ ]:
calfire_school_parcels.to_csv(
    PROCESSED / "schools" / "calfire_school_parcels.csv",
    index=False
)

In [ ]:
calfire_school_major_parcels.to_csv(
    PROCESSED / "schools" / "calfire_school_major_parcels.csv",
    index=False
)

In [ ]:
private_schools = pd.read_csv(
    RAW / "schools" / "Los_Angeles_Private_Schools.csv"
)

private_schools.head()

In [ ]:
private_schools.columns.tolist()

In [ ]:
private_schools["SchoolType"].value_counts()

In [ ]:
private_schools[
    [
        "School",
        "LowGrade",
        "HighGrade"
    ]
].head(20)

In [ ]:
private_schools_gdf = gpd.GeoDataFrame(
    private_schools,
    geometry=gpd.points_from_xy(
        private_schools["X"],
        private_schools["Y"]
    ),
    crs="EPSG:4326"
)

In [ ]:
private_schools_gdf = private_schools_gdf.to_crs(
    perimeter.crs
)

In [ ]:
private_schools_in_fire = gpd.sjoin(
    private_schools_gdf,
    perimeter,
    predicate="intersects"
)

In [ ]:
len(private_schools_in_fire)

In [ ]:
private_schools[["X", "Y"]].head()

In [ ]:
private_schools_gdf = gpd.GeoDataFrame(
    private_schools,
    geometry=gpd.points_from_xy(
        private_schools["Longitude"],
        private_schools["Latitude"]
    ),
    crs="EPSG:4326"
)

In [ ]:
private_schools_gdf = gpd.GeoDataFrame(
    private_schools,
    geometry=gpd.points_from_xy(
        private_schools["longitude"],
        private_schools["latitude"]
    ),
    crs="EPSG:4326"
)

In [ ]:
private_schools_gdf = gpd.GeoDataFrame(
    private_schools,
    geometry=gpd.points_from_xy(
        private_schools["X"],
        private_schools["Y"]
    ),
    crs="EPSG:3857"
)

In [ ]:
private_schools_gdf = private_schools_gdf.to_crs(
    perimeter.crs
)

In [ ]:
private_schools_in_fire = gpd.sjoin(
    private_schools_gdf,
    perimeter,
    predicate="intersects"
)

len(private_schools_in_fire)

In [ ]:
private_schools_gdf.total_bounds

In [ ]:
perimeter.total_bounds


In [ ]:
perimeter_buffer = perimeter.to_crs("EPSG:3857").copy()
perimeter_buffer["geometry"] = perimeter_buffer.geometry.buffer(1000)

private_schools_3857 = private_schools_gdf.to_crs("EPSG:3857")

private_schools_near_fire = gpd.sjoin(
    private_schools_3857,
    perimeter_buffer,
    predicate="intersects"
)

len(private_schools_near_fire)

In [ ]:
private_schools[
    private_schools.astype(str)
    .apply(lambda col: col.str.contains("ALTADENA|PASADENA", case=False, na=False))
    .any(axis=1)
][
    [
        "School",
        "Street",
        "City",
        "X",
        "Y"
    ]
]

In [ ]:
schools_map_export = (
    calfire_school_assessed[
        [
            "Name",
            "* Damage",
            "Latitude",
            "Longitude"
        ]
    ]
    .drop_duplicates(subset=["Name"])
    .copy()
)

In [ ]:
calfire_school_assessed.columns.tolist()


In [ ]:
schools_map_export = (
    calfire_school_assessed[
        [
            "Site Address (parcel)",
            "* Damage",
            "Latitude",
            "Longitude"
        ]
    ]
    .drop_duplicates(subset=["Site Address (parcel)"])
    .copy()
)

schools_map_export = schools_map_export.rename(columns={
    "Site Address (parcel)": "institution_name",
    "* Damage": "damage_class",
    "Latitude": "latitude",
    "Longitude": "longitude"
})

schools_map_export["category"] = "School"
schools_map_export["capacity"] = None

schools_map_export = gpd.GeoDataFrame(
    schools_map_export,
    geometry=gpd.points_from_xy(
        schools_map_export["longitude"],
        schools_map_export["latitude"]
    ),
    crs="EPSG:4326"
)

schools_map_export.to_file(
    PROCESSED / "mapping" / "schools_map.gpkg",
    driver="GPKG"
)